# M53 official MonoDGP reference reproducibility gate

Evaluation only: no training, product-taxonomy adaptation, compression, or architecture change. This notebook pins the official CVPR 2025 MonoDGP source and its best published moderate-Car validation checkpoint, safely loads the public checkpoint with PyTorch's restricted weights-only unpickler, runs a one-batch CUDA smoke test, then evaluates all 3,769 Chen validation images. The gate compares MonoDGP's native official KITTI evaluator with the published 30.1314 / 22.7109 / 19.3978 easy/moderate/hard Car 3D AP_R40 result at a 0.5-AP tolerance; the MobileADAS3D evaluator is retained as an independent diagnostic. A pass authorizes a separate M54 Vehicle/Pedestrian adaptation; it is not a product-safety or iPhone-deployment claim. Run top-to-bottom on a fresh Colab GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, os, shlex, shutil, subprocess, sys
MOBILE_REPO=Path('/content/mobile_adas3d'); MONODGP_REPO=Path('/content/MonoDGP_M53_REF')
MONODGP_COMMIT='aa059a18214aebf644510e7f0793971b403f9d14'
CHECKPOINT_FILE_ID='1nfCiFIxCIm0WG--cbllkqzgeuVRPpNI5'
CHECKPOINT_SHA256='1d5f30b34b8bef49638079a8b07f05ebf11bb5f85d6a9a11c7b028c69396f05d'
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti'); LOCAL_DATASET_ROOT=Path('/content/kitti')
SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen'); DATASET_ROOT=Path('/content/monodgp_kitti_m53')
OUTPUT_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/references/monodgp_m53')
OFFICIAL_CHECKPOINT=OUTPUT_ROOT/'checkpoints/monodgp_val_best_moderate.pth'
def sha256_file(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): digest.update(block)
    return digest.hexdigest()
def run(command,cwd=None,env=None):
    command=[str(x) for x in command]; print('+',shlex.join(command),flush=True)
    merged=os.environ.copy(); merged.update(env or {}); result=subprocess.run(command,cwd=cwd,env=merged)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
run(['nvidia-smi'])


In [ ]:
# Fetch pinned sources, apply only audited compatibility changes, and compile for the active GPU.
if not MOBILE_REPO.exists(): run(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO])
else: run(['git','pull','--ff-only'],cwd=MOBILE_REPO)
if not MONODGP_REPO.exists(): run(['git','clone','https://github.com/PuFanqi23/MonoDGP.git',MONODGP_REPO])
run(['git','fetch','--all'],cwd=MONODGP_REPO); run(['git','checkout',MONODGP_COMMIT],cwd=MONODGP_REPO)
run([sys.executable,'-m','pip','install','-q','gdown','pyyaml','scipy','opencv-python-headless','numba','scikit-image','scikit-learn','tqdm','thop','ninja','pandas'])
run([sys.executable,'scripts/patch_monodgp_colab_compat.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
changed=set(subprocess.run(['git','diff','--name-only'],cwd=MONODGP_REPO,check=True,capture_output=True,text=True).stdout.splitlines())
expected={'lib/helpers/save_helper.py','lib/models/monodgp/ops/modules/ms_deform_attn.py','lib/models/monodgp/ops/setup.py','lib/models/monodgp/ops/src/cuda/ms_deform_attn_cuda.cu'}
if changed != expected: raise RuntimeError(f'Unexpected patched source set: {changed}')
ops=MONODGP_REPO/'lib/models/monodgp/ops'; shutil.rmtree(ops/'build',ignore_errors=True)
run([sys.executable,'setup.py','build','install'],cwd=ops,env={'MAX_JOBS':'2'})
run([sys.executable,'-c','import torch, MultiScaleDeformableAttention; print(torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0))'],cwd=MONODGP_REPO)


In [ ]:
# Create an isolated canonical Chen-split KITTI view from local storage or Google Drive.
def resolve(root,names):
    for name in names:
        path=root/name
        if path.is_dir(): return path
sources={key:resolve(LOCAL_DATASET_ROOT,names) or resolve(DRIVE_DATASET_ROOT,names) for key,names in {'image_2':['training/image_2','training/image_02'],'label_2':['training/label_2','training/label_02'],'calib':['training/calib']}.items()}
if any(path is None for path in sources.values()): raise FileNotFoundError(sources)
(DATASET_ROOT/'training').mkdir(parents=True,exist_ok=True); (DATASET_ROOT/'ImageSets').mkdir(parents=True,exist_ok=True)
for name,target in sources.items():
    link=DATASET_ROOT/'training'/name
    if link.is_symlink() and link.resolve()==target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target,target_is_directory=True)
for split in ('train','val'): shutil.copy2(SPLIT_DIR/f'{split}.txt',DATASET_ROOT/'ImageSets'/f'{split}.txt')
assert len((DATASET_ROOT/'ImageSets/train.txt').read_text().splitlines())==3712
assert len((DATASET_ROOT/'ImageSets/val.txt').read_text().splitlines())==3769


In [ ]:
# Download once to durable Drive storage and reject any checkpoint whose bytes do not match the audited public artifact.
OFFICIAL_CHECKPOINT.parent.mkdir(parents=True,exist_ok=True)
if OFFICIAL_CHECKPOINT.is_file():
    if sha256_file(OFFICIAL_CHECKPOINT)!=CHECKPOINT_SHA256: raise RuntimeError(f'Existing checkpoint hash mismatch: {OFFICIAL_CHECKPOINT}')
else:
    partial=OFFICIAL_CHECKPOINT.with_suffix('.pth.partial')
    if partial.exists(): partial.unlink()
    run([sys.executable,'-m','gdown','--id',CHECKPOINT_FILE_ID,'--output',partial])
    if sha256_file(partial)!=CHECKPOINT_SHA256: raise RuntimeError('Downloaded MonoDGP checkpoint hash mismatch')
    partial.replace(OFFICIAL_CHECKPOINT)
print('Official checkpoint:',OFFICIAL_CHECKPOINT,'size MiB=',round(OFFICIAL_CHECKPOINT.stat().st_size/1024/1024,1),'sha256=',sha256_file(OFFICIAL_CHECKPOINT))


In [ ]:
# Freeze provenance and prepare the official Car-only evaluation. This cell cannot authorize training.
run([sys.executable,'scripts/prepare_monodgp_m53_reference.py','--monodgp-repo',MONODGP_REPO,'--dataset-root',DATASET_ROOT,'--official-checkpoint',OFFICIAL_CHECKPOINT,'--output-root',OUTPUT_ROOT],cwd=MOBILE_REPO)
MANIFEST=OUTPUT_ROOT/'m53_monodgp_reference_manifest.json'; manifest=json.loads(MANIFEST.read_text())
assert manifest['training_authorized'] is False and manifest['product_taxonomy_adaptation'] is False
assert manifest['upstream_commit']==MONODGP_COMMIT and manifest['checkpoint_sha256']==CHECKPOINT_SHA256
print(json.dumps(manifest,indent=2))


In [ ]:
# Mandatory one-batch CUDA preflight using the restricted weights-only checkpoint loader.
SMOKE=OUTPUT_ROOT/'m53_monodgp_reference_smoke.json'; LOG=OUTPUT_ROOT/'colab_logs/m53_smoke.log'; LOG.parent.mkdir(parents=True,exist_ok=True)
command=[sys.executable,'-u','scripts/smoke_test_monodgp_m53_reference.py','--monodgp-repo',MONODGP_REPO,'--manifest',MANIFEST,'--output',SMOKE]
print('+',shlex.join(map(str,command)),'\nDurable log:',LOG,flush=True)
with LOG.open('w',encoding='utf-8',buffering=1) as log:
    process=subprocess.Popen([str(x) for x in command],cwd=MOBILE_REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout: print(line,end='',flush=True); log.write(line)
    code=process.wait()
if code: raise RuntimeError(f'M53 smoke test exited {code}; full log: {LOG}')
smoke=json.loads(SMOKE.read_text()); assert smoke['complete'] and smoke['finite_outputs'] and smoke['safe_weights_only_load']
assert smoke['optimizer_steps']==0
print(json.dumps(smoke,indent=2))


In [ ]:
# Finalize native official AP_R40 reproduction plus an independent diagnostic; complete existing predictions are reused automatically.
LOG=OUTPUT_ROOT/'colab_logs/m53_complete_evaluation.log'
command=[sys.executable,'-u','scripts/evaluate_monodgp_m53_reference.py','--mobile-repo',MOBILE_REPO,'--monodgp-repo',MONODGP_REPO,'--manifest',MANIFEST,'--dataset-root',DATASET_ROOT,'--split-dir',SPLIT_DIR]
print('+',shlex.join(map(str,command)),'\nDurable log:',LOG,flush=True)
with LOG.open('w',encoding='utf-8',buffering=1) as log:
    process=subprocess.Popen([str(x) for x in command],cwd=MOBILE_REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout: print(line,end='',flush=True); log.write(line)
    code=process.wait()
if code: raise RuntimeError(f'M53 evaluation exited {code}; full log: {LOG}')
REPORT=OUTPUT_ROOT/'m53_monodgp_reference_gate.json'; report=json.loads(REPORT.read_text())
print(json.dumps(report,indent=2))
import pandas as pd
display(pd.read_csv(OUTPUT_ROOT/'m53_monodgp_reference_gate.csv'))
print('M54 two-class adaptation authorized:',report['two_class_adaptation_authorized'])
